In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# Подключаем нашу папку src, чтобы брать пути из config.py
sys.path.insert(0, str(Path.cwd().parent))
from src.config import RAW_FILES, REVIEW_SAMPLE_JSONL

# Читаем наш срез отзывов (200 000 строк) и таблицу заведений
df_reviews = pd.read_json(REVIEW_SAMPLE_JSONL, lines=True)
df_business = pd.read_json(RAW_FILES["business"], lines=True)

print("Размер среза отзывов (строк, столбцов):", df_reviews.shape)
display(df_reviews.head(3))

print("Размер таблицы заведений (строк, столбцов):", df_business.shape)
display(df_business.head(3))

Размер среза отзывов (строк, столбцов): (200000, 9)


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30


Размер таблицы заведений (строк, столбцов): (150346, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."


In [2]:
# Сводная таблица по столбцам среза отзывов (review)
review_info = pd.DataFrame({
    "Тип данных": df_reviews.dtypes,
    "Уникальных значений": df_reviews.nunique(),
    "Пропусков (шт)": df_reviews.isna().sum(),
    "Доля пропусков (%)": (df_reviews.isna().mean() * 100).round(2)
})
print("Анализ столбцов таблицы отзывов (review):")
display(review_info)

# Сводная таблица по столбцам заведений (business)
# Столбцы attributes и hours содержат словари, поэтому для nunique переводим их в строку
business_nunique = df_business.astype(str).nunique()
business_info = pd.DataFrame({
    "Тип данных": df_business.dtypes,
    "Уникальных значений": business_nunique,
    "Пропусков (шт)": df_business.isna().sum(),
    "Доля пропусков (%)": (df_business.isna().mean() * 100).round(2)
})
print("Анализ столбцов таблицы заведений (business):")
display(business_info)

Анализ столбцов таблицы отзывов (review):


,Тип данных,Уникальных значений,Пропусков (шт),Доля пропусков (%)
review_id,str,200000,0,0.0
user_id,str,143686,0,0.0
business_id,str,11451,0,0.0
stars,int64,5,0,0.0
useful,int64,59,0,0.0
funny,int64,38,0,0.0
cool,int64,40,0,0.0
text,str,199816,0,0.0
date,datetime64[us],199818,0,0.0


Анализ столбцов таблицы заведений (business):


,Тип данных,Уникальных значений,Пропусков (шт),Доля пропусков (%)
business_id,str,150346,0,0.00
name,str,114117,0,0.00
address,str,122844,0,0.00
city,str,1416,0,0.00
state,str,27,0,0.00
postal_code,str,3362,0,0.00
latitude,float64,135593,0,0.00
longitude,float64,131918,0,0.00
stars,float64,9,0,0.00
review_count,int64,1158,0,0.00


In [3]:
# 1. Числовая статистика (проверяем минимумы и максимумы)
display(df_reviews.describe())
display(df_business.describe())

# 2. Дефект: сколько городов дублируется из-за разного регистра (например, Tampa и tampa) и пробелов
cities_raw = df_business["city"].nunique()
cities_clean = df_business["city"].str.strip().str.lower().nunique()
print(f"Уникальных городов до очистки: {cities_raw}, после очистки: {cities_clean}")
print(f"Найдено дубликатов названий городов: {cities_raw - cities_clean}")

# 3. Дефект: скрытые пропуски (пустые строки "" в адресах и индексах)
empty_address = (df_business["address"] == "").sum()
print(f"Скрытых пропусков (пустых строк '') в столбце address: {empty_address}")

,stars,useful,funny,cool,date
count,200000.000000,200000.000000,200000.000000,200000.000000,200000
mean,3.841425,0.901890,0.257635,0.349225,2015-04-26 21:06:07.436475
min,1.000000,0.000000,0.000000,0.000000,2005-03-01 17:47:15
25%,3.000000,0.000000,0.000000,0.000000,2013-11-17 15:04:42
50%,4.000000,0.000000,0.000000,0.000000,2015-09-25 17:04:52.500000
75%,5.000000,1.000000,0.000000,0.000000,2017-04-04 04:50:32.750000
max,5.000000,320.000000,98.000000,49.000000,2019-01-27 15:08:14
std,1.357044,2.062767,1.002639,1.077559,NaN


,latitude,longitude,stars,review_count,is_open
count,150346.000000,150346.000000,150346.000000,150346.000000,150346.00000
mean,36.671150,-89.357339,3.596724,44.866561,0.79615
std,5.872759,14.918502,0.974421,121.120136,0.40286
min,27.555127,-120.095137,1.000000,5.000000,0.00000
25%,32.187293,-90.357810,3.000000,8.000000,1.00000
50%,38.777413,-86.121179,3.500000,15.000000,1.00000
75%,39.954036,-75.421542,4.500000,37.000000,1.00000
max,53.679197,-73.200457,5.000000,7568.000000,1.00000


Уникальных городов до очистки: 1416, после очистки: 1238
Найдено дубликатов названий городов: 178
Скрытых пропусков (пустых строк '') в столбце address: 5127
